# standard codes

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import requests
import os
import time
from google.colab import userdata
from google.colab import drive
drive.mount('/content/drive')
import json
import glob

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
tng_api_key = userdata.get('TNG_API_KEY')
baseUrl = 'http://www.tng-project.org/api/'
headers = {"api-key":tng_api_key}

In [11]:
def get(path, params=None, out_filename=None):
    headers = {"api-key":tng_api_key}
    r = requests.get(path, params=params, headers=headers)
    r.raise_for_status()

    if out_filename is not None:
        with open(out_filename, 'wb') as f:
            f.write(r.content)
        return out_filename

    if r.headers['content-type'] == 'application/json':
        return r.json()

    if 'content-disposition' in r.headers:
        filename = r.headers['content-disposition'].split("filename=")[1]
        with open(filename, 'wb') as f:
            f.write(r.content)
        return filename

    return r

In [12]:
r = get(baseUrl)

for simulation in r['simulations']: #only get TNG50
    if simulation['name'] == 'TNG50-1':
        url = simulation['url']
        break

tng50 = get(url)

url = 'http://www.tng-project.org/api/TNG50-1/snapshots/z=1.8/'
snapshot = get(url)

#new codes

In [13]:
drive_path = '/content/drive/MyDrive/docs'
os.makedirs(drive_path, exist_ok=True)

In [16]:
files = np.sort(glob.glob(f"{drive_path}/sub*.json"))

for i in range(snapshot['num_groups_subfind']):
    if i % int(snapshot['num_groups_subfind']/1000) == 0:
        print(i)
    if f"{drive_path}/sub{i}.json" in files:
        print("skipping i: ", i)
        continue
    sub_url = f"http://www.tng-project.org/api/TNG50-1/snapshots/{snapshot['number']}/subhalos/{i}/"
    subhalo = get((sub_url), out_filename=f"{drive_path}/sub{i}.json")
    if i > 6:
        break

0
skipping i:  0
skipping i:  1
skipping i:  2
skipping i:  3
skipping i:  4
skipping i:  5
skipping i:  6
skipping i:  7
skipping i:  8
skipping i:  9
skipping i:  10
skipping i:  11
skipping i:  12
skipping i:  13
skipping i:  14
skipping i:  15
skipping i:  16
skipping i:  17
skipping i:  18
skipping i:  19
skipping i:  20
skipping i:  21
skipping i:  22
skipping i:  23
skipping i:  24
skipping i:  25
skipping i:  26
skipping i:  27
skipping i:  28
skipping i:  29
skipping i:  30
skipping i:  31
skipping i:  32
skipping i:  33
skipping i:  34
skipping i:  35
skipping i:  36
skipping i:  37
skipping i:  38
skipping i:  39
skipping i:  40
skipping i:  41
skipping i:  42
skipping i:  43
skipping i:  44
skipping i:  45
skipping i:  46
skipping i:  47
skipping i:  48
skipping i:  49
skipping i:  50
skipping i:  51
skipping i:  52
skipping i:  53
skipping i:  54
skipping i:  55
skipping i:  56
skipping i:  57
skipping i:  58
skipping i:  59
skipping i:  60
skipping i:  61
skipping i:  62


KeyboardInterrupt: 

In [18]:
bhmdot = np.zeros(len(files)) -1
sfr = np.zeros(len(files)) -1

print('bhmdot: ', bhmdot)
print("sfr: ", sfr)

save_file = f"{drive_path}/bh_sfr.npz"
if os.path.exists(save_file):
    data = np.load(save_file)
    bhmdot[:len(data['bhmdot'])] = data['bhmdot']
    sfr[:len(data['sfr'])] = data['sfr']

print('bhmdot: ', bhmdot)
print("sfr: ", sfr)

for i in range(len(files)):
    if bhmdot[i] != -1:
        continue
    if sfr[i] != -1:
        continue
    with open(files[i], 'r') as fid:
        data = json.load(fid)
    bhmdot[i] = data['bhmdot']
    sfr[i] = data['sfr']
    if i > 2:
        break
    np.savez_compressed(save_file, bhmdot=bhmdot, sfr=sfr)

bhmdot:  [-1. -1. -1. ... -1. -1. -1.]
sfr:  [-1. -1. -1. ... -1. -1. -1.]
bhmdot:  [ 3.51652e-03  1.66792e-02  5.11530e-04 ... -1.00000e+00 -1.00000e+00
 -1.00000e+00]
sfr:  [410.991    61.4731    1.57871 ...  -1.       -1.       -1.     ]
